# 01 · Understand a race weekend

**Goal:** read the session, driver, and lap tables from a real Formula 1 weekend.

Your task is to understand the tables and count each driver's unique laps. This notebook prepares the data; we will make our first prediction in notebook 02. Example: the 2024 Bahrain Grand Prix.

## Setup

Select **WeAreChecking (.venv)** as the kernel at the top right. Use this project's environment.
If you need to download the data, run the fetch command in `START_HERE.md` from the terminal.
Run the cells in order with `Shift+Enter`.

In [ ]:
import importlib
import pandas as pd
from IPython.display import display
from wearechecking.paths import project_root
from wearechecking.data import latest_snapshot
from wearechecking import student
from wearechecking.learning import check_lap_counts

ROOT = project_root()
YEAR, ROUND = 2024, 1
print('WeAreChecking environment loaded.')

## Steps

### 1. Open the snapshot

A snapshot is a saved copy of the downloaded data that we preserve unchanged.
`observed_at` records when we obtained the data, which differs from the date of the race.
If the data is missing, this cell shows the fetch command you need to run.

In [ ]:
snapshot = latest_snapshot(YEAR, ROUND, root=ROOT)
print('Event:', snapshot.manifest['event_name'])
print('Source:', snapshot.manifest['source'])
print('Observed at:', snapshot.manifest['observed_at'])
print('Provenance mode:', snapshot.manifest['provenance_mode'])

### 2. Read the session schedule

Each row represents **one session**. FP1/FP2/FP3 are free practice, Q is qualifying, and R is the race.
Sprint weekends can include SQ and S. We read the sessions from the schedule.
All times are UTC. Describe the order of the sessions in your own words.

In [ ]:
schedule = snapshot.schedule()
display(schedule)
expected = set(schedule.session_code)
loaded = set(snapshot.manifest['sessions'])
assert loaded == expected, f'Missing sessions: {expected - loaded}. Use fetch to download the full weekend.'
print(f'{len(loaded)} / {len(expected)} sessions loaded.')

### 3. Read the lap table

Each row represents **one driver's lap record**. `DriverNumber` identifies a driver within the session; it is not a permanent driver ID.
`LapTime` can be missing, for example on an opening lap or when timing data is unavailable. Keep missing values distinct from zero.

**Pause and explain:** What is the difference between DriverNumber and LapNumber?

In [ ]:
laps = snapshot.table('FP1', 'laps')
display(laps.sort_values(['DriverNumber', 'LapNumber'])[['DriverNumber', 'LapNumber', 'LapTime', 'is_pit_out_lap']].head(8))
print('Rows:', len(laps))
print('Unique driver numbers:', laps.DriverNumber.nunique())
print('Missing lap times:', int(laps.LapTime.isna().sum()))

### 4. Understand the other tables

The participants table tells us who entered this session. Stints describe which laps were driven on each tyre set. Weather records describe the conditions observed at the time.
We will decide which information belongs in the model when we define its prediction time and target.

In [ ]:
overview = pd.DataFrame([
    {'table': name, 'rows': len(snapshot.table('FP1', name))}
    for name in ['participants', 'laps', 'stints', 'weather']
])
display(overview)

### 5. Your exercise: lap_counts

Edit `lap_counts` in `src/wearechecking/student.py`.

Output columns: **driver_number, lap_count**, with one row per driver.
Count duplicate `(DriverNumber, LapNumber)` records only once. Exclude zero or missing lap numbers.

Hints:

1. Select rows with valid lap numbers.
2. Which two columns together define a unique lap for `drop_duplicates`?
3. Use `groupby` to group rows by driver, then count the rows.
4. Give the output columns their required names.

We will build the solution one small step at a time.

In [ ]:
importlib.reload(student)
answer = student.lap_counts(laps.copy())
print(check_lap_counts(answer, laps))
if answer is not None:
    display(answer.head(10))

## Checks

- I can identify the data source and observation time.
- The snapshot contains every scheduled session.
- I can distinguish a lap record from a driver record.
- The `lap_counts` check reports **PASSED**.
- I can explain why completing more laps does not necessarily mean being faster.

Milestone 1 is complete only after the last two points are satisfied.

## Next steps

Share your explanation, and we will implement `lap_counts` together. Then we will move to notebook 02.

Sources: [OpenF1](https://openf1.org/docs/) · [Jolpica](https://github.com/jolpica/jolpica-f1/blob/main/docs/README.md).